# 03 — Borrow Rate Dynamics & Factor Analysis

## Overview

This notebook addresses two related questions:

1. **Borrow cost structure**: How do proxy borrow rates vary cross-sectionally (by utilisation level) and over time?  Does the proxy assign intuitively correct rates to known hard-to-borrow (HTB) names?

2. **Fama-MacBeth regression**: Does the short-interest / borrow-stress signal earn positive risk-adjusted returns after controlling for standard factor exposures (size, momentum, volatility)?

### Key hypothesis
High *borrow stress* (proxy rate > sector 90th percentile) should carry a negative return premium: short sellers face high carrying costs, so only the most informed sellers hold these positions → informed short pressure predicts negative returns.

In [ ]:
import sys; sys.path.insert(0, "../src")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from securities_lending.features import BorrowRateProxy
from securities_lending.analysis import FamaMacBeth
from securities_lending.viz import plot_borrow_proxy_calibration, plot_fm_coefficients

plt.rcParams.update({"figure.facecolor": "white", "axes.grid": True, "grid.alpha": 0.3})

features = pd.read_parquet("../data/processed/features.parquet")
features["date"] = pd.to_datetime(features["date"])
print(f"{len(features):,} rows, {features['symbol'].nunique()} tickers")

## 1. Borrow Rate Proxy — Calibration and Diagnostics

The proxy model maps estimated utilisation (short_interest / lendable_supply) to an annualised borrow fee via a piecewise nonlinear schedule.  Here we:
1. Plot the proxy rate curve
2. Check proxy rates for known HTB names
3. Examine the cross-sectional distribution by borrow tier

In [ ]:
# Visualise the piecewise rate schedule
proxy_model = BorrowRateProxy(lendable_fraction=0.20)
fig, ax = plt.subplots(figsize=(8, 4))
proxy_model.plot_rate_curve(ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# Cross-sectional analysis of proxy borrow rates
if "borrow_rate_bps" in features.columns:
    borrow_df = features[["date", "symbol", "utilisation", "borrow_rate_bps",
                           "borrow_rate_z", "borrow_stress"]].dropna(subset=["borrow_rate_bps"])

    # Known HTB validation: these names are widely cited as hard-to-borrow
    known_htb = {"GME": 300, "AMC": 150, "BYND": 200, "SPCE": 100,
                 "MSTR": 80, "COIN": 60}

    fig = plot_borrow_proxy_calibration(borrow_df, known_htb=known_htb)
    plt.show()

    # Summary statistics by tier
    latest = borrow_df.sort_values("date").groupby("symbol").last().reset_index()
    print("
Borrow tier breakdown (latest cross-section):")
    bins = [0, 50, 150, 500, 2000, np.inf]
    labels = ["ETB (<50bps)", "50–150bps", "150–500bps", "HTB (500–2000bps)", "Special (>2000bps)"]
    latest["tier"] = pd.cut(latest["borrow_rate_bps"], bins=bins, labels=labels)
    print(latest["tier"].value_counts().to_string())

## 2. Borrow Rate Time Series — Selected Names

For a few representative tickers spanning the borrow rate distribution, plot proxy borrow rate alongside price to visualise borrow pressure dynamics.

In [ ]:
focus = ["GME", "AMC", "AAPL", "SPY"]
if "borrow_rate_bps" in features.columns:
    fig, axes = plt.subplots(len(focus), 1, figsize=(12, 3*len(focus)), sharex=True)
    for ax, sym in zip(axes, focus):
        sub = features[features["symbol"] == sym].set_index("date")
        if sub.empty:
            continue
        ax2 = ax.twinx()
        sub["borrow_rate_bps"].plot(ax=ax, color="#C53030", alpha=0.8, linewidth=1)
        if "close" in sub.columns:
            sub["close"].plot(ax=ax2, color="#2B6CB0", linewidth=1.5, alpha=0.6)
        ax.set_ylabel("Borrow rate (bps)", color="#C53030", fontsize=9)
        ax2.set_ylabel("Price (USD)", color="#2B6CB0", fontsize=9)
        ax.set_title(sym)
    fig.suptitle("Proxy Borrow Rate vs Price", fontsize=12)
    plt.tight_layout()
    plt.show()

## 3. Fama-MacBeth Regression

We test whether borrow stress and short interest signals earn returns *after* controlling for:
- : size factor
- : short-term reversal
- : volatility factor

The FM regression standardises all variables cross-sectionally before running each date-specific regression, so coefficients are directly interpretable as returns per one cross-sectional standard deviation.

In [ ]:
def pivot(df, col):
    return df.pivot(index="date", columns="symbol", values=col)

ret5 = pivot(features, "ret_fwd_5d")
SIGNALS = ["svr_z20", "si_pct_float", "borrow_rate_bps", "borrow_stress", "short_pressure"]
CONTROLS = ["realized_vol_20d"]

fm_results = {}
for sig in SIGNALS:
    if sig not in features.columns:
        continue
    sig_panel = pivot(features, sig).dropna(how="all", axis=1)
    controls = {c: pivot(features, c) for c in CONTROLS if c in features.columns}
    try:
        fm = FamaMacBeth(
            forward_return_panel=ret5,
            signal_panel=sig_panel,
            control_panels=controls,
            nw_lags=4,
        )
        res = fm.run(signal_name=sig, horizon=5)
        fm_results[sig] = res
        print(f"
{sig}:")
        print(res.summary().to_string())
    except Exception as e:
        print(f"{sig}: {e}")

In [ ]:
# Visualise FM coefficients with Newey-West confidence intervals
if fm_results:
    from securities_lending.viz import plot_fm_coefficients
    fig = plot_fm_coefficients(fm_results, use_nw=True)
    plt.show()

## 4. Incremental R² — Does SI/Borrow Add Beyond Standard Controls?

For each signal, we test whether it adds incremental cross-sectional R² beyond the control variables alone.

In [ ]:
if fm_results:
    print("Incremental R^2 Test (signal vs controls-only):")
    for sig, res in fm_results.items():
        sig_panel = pivot(features, sig).dropna(how="all", axis=1)
        controls = {c: pivot(features, c) for c in CONTROLS if c in features.columns}
        fm = FamaMacBeth(
            forward_return_panel=ret5,
            signal_panel=sig_panel,
            control_panels=controls,
        )
        incr = fm.compare_incremental(signal_name=sig, horizon=5)
        print(f"  {sig:<25s}  ΔR²={incr.get('delta_r2_mean','N/A'):.5f}  t={incr.get('t_stat','N/A'):.2f}  p={incr.get('p_value','N/A'):.4f}")